# Scraping Stock Prices from HTML Tables

Two ways to turn a web page's stock-price table into a DataFrame: looping through the HTML with BeautifulSoup, and the one-line `pd.read_html()`. Worked example on Netflix, then the same on Amazon.

*Practice lab from IBM's **Python Project for Data Science** course (Coursera), documented in my own words.*

## Libraries

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [3]:
import warnings
# Hide FutureWarnings to keep the output readable
warnings.filterwarnings("ignore", category=FutureWarning)

## Netflix: the BeautifulSoup way

Download the page, parse the HTML, then walk the table body row by row, pulling each cell's text into a DataFrame.

In [4]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/netflix_data_webpage.html"

In [5]:
data  = requests.get(url).text

In [6]:
soup = BeautifulSoup(data, 'html.parser')

In [7]:
netflix_data = pd.DataFrame(columns=["Date", "Open", "High", "Low", "Close", "Volume"])

In [10]:
# First we isolate the body of the table which contains all the information
# Then we loop through each row and find all the column values for each row
for row in soup.find("tbody").find_all('tr'):
    col = row.find_all("td")
    date = col[0].text
    Open = col[1].text
    high = col[2].text
    low = col[3].text
    close = col[4].text
    adj_close = col[5].text
    volume = col[6].text
    
    # Finally we append the data of each row to the table
    netflix_data = pd.concat([netflix_data,pd.DataFrame({"Date":[date], "Open":[Open], "High":[high], "Low":[low], "Close":[close], "Adj Close":[adj_close], "Volume":[volume]})], ignore_index=True)


In [11]:
netflix_data.head()

,Date,Open,High,Low,Close,Volume,Adj Close
0,"Jun 01, 2021",504.01,536.13,482.14,528.21,"78,560,600",528.21
1,"May 01, 2021",512.65,518.95,478.54,502.81,"66,927,600",502.81
2,"Apr 01, 2021",529.93,563.56,499.00,513.47,"111,573,300",513.47
3,"Mar 01, 2021",545.57,556.99,492.85,521.66,"90,183,900",521.66
4,"Feb 01, 2021",536.79,566.65,518.28,538.85,"61,902,300",538.85


## Netflix: the pandas way

`pd.read_html()` finds every `<table>` on a page and returns a list of DataFrames. This page has one table, so I take the first.

In [12]:
read_html_pandas_data = pd.read_html(url)

In [15]:
from io import StringIO

read_html_pandas_data = pd.read_html(StringIO(str(soup)))

In [17]:
netflix_dataframe = read_html_pandas_data[0]

netflix_dataframe.head()

,Date,Open,High,Low,Close*,Adj Close**,Volume
0,"Jun 01, 2021",504.01,536.13,482.14,528.21,528.21,78560600
1,"May 01, 2021",512.65,518.95,478.54,502.81,502.81,66927600
2,"Apr 01, 2021",529.93,563.56,499.00,513.47,513.47,111573300
3,"Mar 01, 2021",545.57,556.99,492.85,521.66,521.66,90183900
4,"Feb 01, 2021",536.79,566.65,518.28,538.85,538.85,61902300


## Amazon: on my own

Same technique on a different page: download, parse, loop through the rows.

In [18]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/amazon_data_webpage.html"
html_data = requests.get(url, timeout=10).text

In [19]:
soup = BeautifulSoup(html_data, 'html.parser')

In [20]:
soup.title

<title>Amazon.com, Inc. (AMZN) Stock Historical Prices &amp; Data - Yahoo Finance</title>

In [25]:
amazon_data = pd.DataFrame(columns=["Date", "Open", "High", "Low", "Close","Adj Close", "Volume"])
rows = soup.tbody.find_all("tr")
for row in rows:
    col = row.find_all("td")
    if col:
        df = pd.DataFrame([
            {"Date": col[0].text.strip(), 
             "Open": col[1].text.strip(), 
             "High": col[2].text.strip(), 
             "Low": col[3].text.strip(), 
             "Close": col[4].text.strip(), 
             "Adj Close": col[5].text.strip(), 
             "Volume": col[6].text.strip()}])
        amazon_data = pd.concat([df, amazon_data], ignore_index=True)
amazon1 = amazon_data

In [28]:
amazon1.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,"Jan 01, 2016",656.29,657.72,547.18,587.00,587.00,"130,200,900"
1,"Feb 01, 2016",578.15,581.80,474.00,552.52,552.52,"124,144,800"
2,"Mar 01, 2016",556.29,603.24,538.58,593.64,593.64,"94,009,500"
3,"Apr 01, 2016",590.49,669.98,585.25,659.59,659.59,"78,464,200"
4,"May 01, 2016",663.92,724.23,656.00,722.79,722.79,"90,614,500"


## What I took away

- BeautifulSoup gives full control over messy pages; `pd.read_html()` is the shortcut whenever the data is already a proper `<table>`.
- Everything scraped arrives as text — prices with commas and volumes like `78,560,600` need cleaning before any maths.